# Q3 : le "début" de l'entraînement se compte-t-il en epochs ou en itérations ? (Colab, GPU)

Le papier règle la durée de l'early dropout en **epochs**, mais ce que le modèle "voit", ce sont des **itérations** (des pas de gradient). Avec un batch de 128, une epoch = 391 itérations (50 000 images d'entraînement). Si on change la taille de batch en gardant 50 epochs, le nombre d'itérations par epoch change, et les deux façons de compter ne disent plus la même chose. C'est ce qui permet de les départager.

**Le principe** : on refait la grille de durées du MLP largeur 16 (50 epochs, p = 0.1) avec un batch **4 fois plus petit (32)** et **4 fois plus grand (512)**. Le batch 128 est déjà fait (grille principale de `fashion_mnist.ipynb`). Tout le reste est identique : mêmes seeds, même split, Adam lr 1e-3, cosinus sur 50 epochs. On ne change qu'une chose à la fois, donc on ne corrige pas le learning rate selon le batch.

| batch | itérations par epoch | 1 epoch de dropout = |
|---|---|---|
| 32 | 1 563 | 4 epochs à batch 128 |
| 128 | 391 | (référence) |
| 512 | 98 | 1/4 d'epoch à batch 128 |

**Ce que prédit chaque hypothèse** (à batch 128, la baisse de loss d'entraînement est forte de 1 à 5 epochs de dropout, faiblit à 10 et disparaît à 15) :
- **Si ça se compte en epochs** : la courbe "écart de loss d'entraînement selon la durée en epochs" est la même pour les trois batchs.
- **Si ça se compte en itérations** : à batch 512 l'effet se décale vers des durées 4 fois plus longues en epochs (fort à 10-15 epochs, faible à 1) ; à batch 32 il se décale vers des durées 4 fois plus courtes (5 epochs à batch 32 = 20 epochs à batch 128 : plus d'effet, voire une perte).

**Les deux comparaisons décisives, fixées avant de lancer** (mesure : loss d'entraînement finale, écart avec sans dropout, même seed) :
1. Batch 512, early de **15 epochs** (frac 0.3) : pas d'effet si c'est en epochs, baisse nette si c'est en itérations.
2. Batch 32, early de **5 epochs** (frac 0.1) : baisse nette si c'est en epochs, plus rien si c'est en itérations.

On vérifie aussi le régime à chaque batch (sans dropout contre dropout classique) : un batch plus petit fait plus de pas de gradient, donc le MLP pourrait ne plus sous-apprendre à batch 32. On a vu que la baisse de loss d'entraînement existe aussi hors sous-apprentissage (contrôle), donc la question epochs/itérations garde un sens, mais il faudra le dire.

**Volume** : MLP, 8 durées × 5 seeds × 2 batchs = 80 runs (environ 30-40 min sur T4, surtout à cause du batch 32). Le CNN est en option à la fin (120 runs, plusieurs heures) : son effet n'est visible qu'avec 10 seeds et n'a pas passé la correction de Holm, donc on commence par le MLP, où l'effet est solide.

Les runs déjà faits sont sautés : si Colab coupe, on relance tout. Les CSV vont dans `resultats/batch32/` et `resultats/batch512/`.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
# si le pull refuse à cause de modifs locales : !git stash  puis relancer la cellule
!git pull

In [ ]:
import os, math, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)
from config import BATCH, P, EARLY_GRID, SEEDS
from models import mlp, cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "il faut un GPU : Exécution > Modifier le type d'exécution"
print(device)

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr, Xte = tr.data.float().unsqueeze(1) / 255, te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split et même normalisation que dans fashion_mnist.ipynb
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

images = {k: v.to(device) for k, v in dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte).items()}   # CNN
vecteurs = {k: (v.flatten(1) if k.startswith("X") else v) for k, v in images.items()}                        # MLP

for b in [32, 128, 512]:
    print(f"batch {b:4d} : {math.ceil(len(Xtr) / b)} itérations par epoch")

In [ ]:
# mêmes fonctions que dans fashion_mnist.ipynb, avec la taille de batch en plus.
# batch 128 -> resultats/ (les runs existent déjà), autre batch -> resultats/batch{b}/
CADRES = {
    "mlp_fmnist": (lambda L: mlp(784, 10, couche_cache=L), vecteurs),
    "cnn_fmnist": (lambda L: cnn(1, 10, largeur=L), images),
}

def dossier(batch):
    return "resultats" if batch == BATCH else f"resultats/batch{batch}"

def fichier(cadre, L, E, frac, seed, batch=BATCH):
    return f"{dossier(batch)}/{cadre}_w{L}_e{E}_frac{frac}_s{seed}.csv"

def lancer(cadre, L, E, frac, seed, batch=BATCH):
    f = fichier(cadre, L, E, frac, seed, batch)
    if os.path.exists(f):
        return                                   # déjà fait
    os.makedirs(dossier(batch), exist_ok=True)
    creer, data = CADRES[cadre]
    torch.manual_seed(seed)                      # même seed = mêmes poids de départ
    model, opt = creer(L)
    model.to(device)
    hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=P, batch=batch)
    hist.assign(cadre=cadre, width=L, epochs=E, frac=frac, seed=seed).to_csv(f, index=False)
    print(f, " val acc", round(hist.val_acc.tail(5).mean(), 4))

## 1. MLP largeur 16, batchs 32 et 512 (80 runs)

Mêmes durées que la grille principale : 0 (sans dropout), 1, 2, 5, 10, 15, 25 epochs, et 50 (dropout classique).

In [ ]:
DUREES = [0, 0.02] + EARLY_GRID[1:]          # 0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1
BATCHS = [512, 32]                           # 512 d'abord : beaucoup plus rapide

for batch in BATCHS:
    for frac in DUREES:
        for seed in SEEDS:
            lancer("mlp_fmnist", 16, 50, frac, seed, batch)

## 2. (Option) CNN largeur 4, batchs 32 et 512, 10 seeds (120 runs)

À ne lancer qu'après la partie 1, si on a le temps. Sur le CNN, l'effet (à batch 128) est à 10-15 epochs de dropout, et seulement visible avec 10 seeds. Les prédictions : en epochs, toujours 10-15 epochs ; en itérations, 2-4 epochs à batch 32 et 40 epochs ou plus (donc hors grille) à batch 512.

In [ ]:
FAIRE_CNN = False                            # passer à True pour lancer

if FAIRE_CNN:
    for batch in BATCHS:
        for frac in [0, 0.05, 0.1, 0.2, 0.3, 1]:
            for seed in range(10):
                lancer("cnn_fmnist", 4, 50, frac, seed, batch)

## Vérification

In [ ]:
for batch in [128] + BATCHS:
    n = sum(os.path.exists(fichier("mlp_fmnist", 16, 50, f, s, batch)) for f in DUREES for s in SEEDS)
    print(f"MLP batch {batch:4d} : {n:3d} / {len(DUREES) * len(SEEDS)}")
for batch in BATCHS:
    n = sum(os.path.exists(fichier("cnn_fmnist", 4, 50, f, s, batch)) for f in [0, 0.05, 0.1, 0.2, 0.3, 1] for s in range(10))
    print(f"CNN batch {batch:4d} : {n:3d} / 60 (option)")

## Premier coup d'œil

L'analyse complète ira dans `fashion_mnist.ipynb`. Ici : l'écart de loss d'entraînement avec "sans dropout" (même seed), en fonction de la durée du dropout, comptée **en epochs** (à gauche) puis **en itérations** (à droite). La bonne façon de compter est celle où les trois courbes se superposent.

In [ ]:
T95 = {5: 2.78, 10: 2.26}

def fin_de_run(f):
    return pd.read_csv(f).tail(5)[["train_loss", "val_loss", "val_acc", "test_acc"]].mean()

def ecarts(cadre, L, batch, fracs, seeds):
    # écart moyen avec "sans dropout" (même seed et même batch) et IC à 95 %, pour chaque durée
    lignes = {}
    for frac in fracs:
        d = pd.DataFrame([fin_de_run(fichier(cadre, L, 50, frac, s, batch)) - fin_de_run(fichier(cadre, L, 50, 0, s, batch))
                          for s in seeds])
        d[["val_acc", "test_acc"]] *= 100
        ic = T95[len(seeds)] * d.std() / len(seeds) ** 0.5
        lignes[frac] = pd.concat([d.mean(), ic.add_prefix("ic_")])
    colonnes = [c for m in ["train_loss", "val_loss", "val_acc", "test_acc"] for c in (m, "ic_" + m)]
    t = pd.DataFrame(lignes).T[colonnes]
    t.insert(0, "epochs_dropout", [round(f * 50) for f in t.index])
    t.insert(1, "iterations_dropout", t.epochs_dropout * math.ceil(len(Xtr) / batch))
    return t.round(3)

tableaux = {b: ecarts("mlp_fmnist", 16, b, DUREES[1:], SEEDS) for b in [32, 128, 512]}
pd.concat({f"batch {b}": t for b, t in tableaux.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for b, t in tableaux.items():
    t = t.drop(index=1)                                    # le dropout classique n'est pas un early
    for ax, x in zip(axes, ["epochs_dropout", "iterations_dropout"]):
        ax.errorbar(t[x], t.train_loss, yerr=t.ic_train_loss, marker="o", capsize=4, label=f"batch {b}")
for ax, titre in zip(axes, ["durée du dropout en epochs", "durée du dropout en itérations"]):
    ax.set_xscale("log"); ax.axhline(0, color="gray", lw=1)
    ax.set_xlabel(titre); ax.set_ylabel("écart de loss d'entraînement (négatif = mieux)"); ax.legend()
plt.tight_layout(); plt.show()

# les deux comparaisons fixées à l'avance, et le régime à chaque batch
for b, frac, nom in [(512, 0.3, "batch 512, early 15 epochs"), (32, 0.1, "batch 32, early 5 epochs")]:
    r = tableaux[b].loc[frac]
    print(f"{nom:28s} écart train loss {r.train_loss:+.4f} ± {r.ic_train_loss:.4f}")
for b, t in tableaux.items():
    r = t.loc[1]
    print(f"régime batch {b:4d} : dropout classique {r.val_acc:+.2f} ± {r.ic_val_acc:.2f} pt de validation (sous-apprentissage si <= -0.5)")

## Sauvegarde sur GitHub (token dans les secrets Colab, jamais dans le code)

In [ ]:
!git config user.name "Felix-prjcts"
!git add -A resultats/
!git commit -m "Q3 epochs vs iterations : MLP w16 e50 a batch 32 et 512"

token = userdata.get("GITHUB_TOKEN")
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git HEAD 2>&1 | sed "s/{token}/***/g"